In [ ]:
import sys
import cv2
from matplotlib import pyplot as plt
import numpy as np
import torch
from torch.nn import functional as F
import kornia.geometry.subpix.dsnt as dsnt
from kornia.utils.grid import create_meshgrid
import math
from einops.einops import rearrange
from typing import Sequence
import importlib
from utils.misc import setup_gpus
from default_config import get_cfg_defaults
from maff.maff import MAFF
from maff.utils.supervision import compute_supervision_coarse, compute_supervision_fine
from utils.metrics import (
    compute_symmetrical_epipolar_errors,
    compute_pose_errors,
    aggregate_metrics,
)
from datasets.overall_dataset import MAFF_Dataset

In [ ]:
latest_ckpt_path = "logs/tb_logs/MegaDepth_640_(0, 0, 1, 1, 1)_8_2_M2_VMamba_T_FPN_PMF/version_0"
latest_ckpt = "checkpoints/last.ckpt"

sys.path.append(latest_ckpt_path)
get_cfg_defaults = importlib.import_module("config").get_cfg_defaults

In [ ]:
config = get_cfg_defaults()
config.DEVICE.GPU_IDX = "7,"
config.DEVICE.ENABLE_DDP = False
# setup exact gpus available and set CUDA_VISIBLE_DEVICES variable
n_gpu_available = (
    setup_gpus(config.DEVICE.GPU_IDX) if config.DEVICE.ENABLE_GPU else 0
)
config.TRAINER.WORLD_SIZE = n_gpu_available * config.DEVICE.NUM_NODES
config.TRAINER.TRUE_BATCH_SIZE = (
    config.TRAINER.WORLD_SIZE * config.LOADER.BATCH_SIZE
)
config.TRAINER.SCALING = (
    config.TRAINER.TRUE_BATCH_SIZE / config.TRAINER.CANONICAL_BS
)
config.TRAINER.TRUE_LR = config.TRAINER.CANONICAL_LR * config.TRAINER.SCALING
config.TRAINER.WARMUP_STEP = math.floor(
    config.TRAINER.WARMUP_STEP / config.TRAINER.SCALING
)
device = torch.device("cuda:0")

In [ ]:
maff = MAFF(config=config["MODEL"])
maff.load_state_dict(torch.load(latest_ckpt_path+"/"+latest_ckpt)["state_dict"])
maff.to(device)

In [ ]:
data_module = MAFF_Dataset(config=config)
data_module.setup(stage="fit")
train_dataloader = data_module.train_dataloader()
val_dataloader = data_module.val_dataloader()

In [ ]:
datas = []

for data in train_dataloader:
    datas.append(data)
    if len(datas) == 20:
        break

In [ ]:
def recursive_to(item, device):
    if isinstance(item, torch.Tensor):
        return item.to(device)
    elif isinstance(item, dict):
        return {key: recursive_to(value, device) for key, value in item.items()}
    elif isinstance(item, list):
        return [recursive_to(value, device) for value in item]
    elif isinstance(item, tuple):
        return tuple(recursive_to(value, device) for value in item)
    else:
        return item

In [ ]:
config.TRAINER.RANSAC_CONF = 0.99999
config.TRAINER.RANSAC_PIXEL_THR = 0.2

In [ ]:
def compute_auc(errors, thresholds):
    """
    计算给定误差的 AUC（Area Under the Curve）
    
    Args:
        errors (torch.Tensor): 误差张量
        thresholds (list): AUC 计算的阈值列表
    
    Returns:
        dict: 不同阈值下的 AUC 值
    """
    errors = errors.cpu().numpy()
    sort_idx = np.argsort(errors)
    errors = errors[sort_idx]
    recall = (np.arange(len(errors)) + 1) / len(errors)
    errors = np.r_[0., errors]
    recall = np.r_[0., recall]
    
    aucs = {}
    for t in thresholds:
        last_index = np.searchsorted(errors, t)
        r = np.r_[recall[:last_index], recall[last_index-1]]
        e = np.r_[errors[:last_index], t]
        aucs[f'auc@{t}'] = np.trapz(r, x=e)/t
    
    return aucs

In [ ]:
for data in datas:
    data = recursive_to(data, device)
    compute_supervision_coarse(data, coarse_scale=config.MODEL.COARSE_SCALE)
    W = data["image0"].shape[-1] / config.MODEL.COARSE_SCALE
    scale = (
        1
        if config["MODEL"]["PIXEL_SHUFFLE_REFINEMENT"]
        else config["MODEL"]["FINE_SCALE"]
    )
    scale0 = data["scale0"] if "scale0" in data else 1
    scale1 = data["scale1"] if "scale1" in data else 1
    # coarse_coord_0 = torch.tensor([[i%W + 0.5,i//W + 0.5] for i in data["spv_i_ids"]], device=device)
    # coarse_coord_1 = torch.tensor([[i%W + 0.5,i//W + 0.5] for i in data["spv_j_ids"]], device=device)
    coarse_coord_0 = torch.tensor([[i%W,i//W] for i in data["spv_i_ids"]], device=device)
    coarse_coord_1 = torch.tensor([[i%W,i//W] for i in data["spv_j_ids"]], device=device)
    
    data.update({
        "b_idx_c": data["spv_b_ids"],
        "i_idx_c": data["spv_i_ids"],
        "j_idx_c": data["spv_j_ids"],
    })
    
    compute_supervision_fine(data, config)
    
    fine_coord_0 = coarse_coord_0 * config.MODEL.COARSE_SCALE * scale0
    # 添加随机扰动
    random_noise = torch.randn_like(data["coord_offset_f_gt"], device=device) * 0.  # 0.1 是扰动的强度，可以根据需要调整
    fixed_noise = torch.tensor([[0.0, 0.0]], device=device).repeat(data["coord_offset_f_gt"].shape[0],1)
    radius = (
        config["MODEL"]["COARSE_SCALE"] // 2
        if config["MODEL"]["PIXEL_SHUFFLE_REFINEMENT"]
        else int(config["MODEL"]["COARSE_SCALE"] // config["MODEL"]["FINE_SCALE"]) // 2
    )
    mask = torch.linalg.norm(data["coord_offset_f_gt"], ord=float("inf"), dim=1) < 1.0
    data["coord_offset_f_gt"][~mask] = 0
    fine_coord_1 = coarse_coord_1 * config.MODEL.COARSE_SCALE * scale1 + (data["coord_offset_f_gt"] + random_noise + fixed_noise) * scale1 * scale * radius

    data.update({"coarse_coord_0": coarse_coord_0, "coarse_coord_1": coarse_coord_1, "fine_coord_0": fine_coord_0, "fine_coord_1": fine_coord_1})

    # 从 data 中获取图像并转换为 numpy 数组
    image0 = data["image0"].detach().cpu().numpy()[0, 0] * 255
    W1 = data["image0"].shape[-1] 
    image1 = data["image1"].detach().cpu().numpy()[0, 0] * 255

    # 确保图像数据类型为 uint8
    image0 = image0.astype(np.uint8)
    image1 = image1.astype(np.uint8)

    # 将灰度图像转换为 BGR 图像
    image0 = cv2.cvtColor(image0, cv2.COLOR_GRAY2RGB)
    image1 = cv2.cvtColor(image1, cv2.COLOR_GRAY2RGB)

    # 将图像插值至两倍大小
    image0 = cv2.resize(image0, (image0.shape[1] * 2, image0.shape[0] * 2), interpolation=cv2.INTER_LINEAR)
    image1 = cv2.resize(image1, (image1.shape[1] * 2, image1.shape[0] * 2), interpolation=cv2.INTER_LINEAR)
    W1 *= 2

    # 将两张图像水平拼接在一起
    combined_image = cv2.hconcat([image0, image1])

    # 预定义一组常规的颜色
    colors = [
        (255, 0, 0),    # 红色
        (0, 255, 0),    # 绿色
        (0, 0, 255),    # 蓝色
        (255, 255, 0),  # 黄色
        (255, 0, 255),  # 洋红
        (0, 255, 255),  # 青色
        (128, 0, 0),    # 深红
        (0, 128, 0),    # 深绿
        (0, 0, 128),    # 深蓝
        (128, 128, 0),  # 橄榄绿
        (128, 0, 128),  # 紫色
        (0, 128, 128),  # 青绿色
        (192, 192, 192),# 银色
        (128, 128, 128),# 灰色
        (0, 0, 0),      # 黑色
        (255, 165, 0),  # 橙色
        (255, 20, 147), # 深粉色
        (75, 0, 130),   # 靛蓝色
        (240, 230, 140),# 卡其色
        (173, 216, 230) # 淡蓝色
    ]

    # 在图像上绘制连线和圆圈
    lines = []
    circles = []

    for idx, ((x0, y0), (x1, y1)) in enumerate(zip(fine_coord_0, fine_coord_1)):
        # 从预定义的颜色中随机选择一个颜色
        color = (np.random.randint(0, 256), np.random.randint(0, 256), np.random.randint(0, 256))
        
        scale0 = data["scale0"][0] if "scale0" in data else 1
        scale1 = data["scale1"][0] if "scale1" in data else 1
        
        # 调整坐标至两倍大小
        x0, y0, x1, y1 = x0 * 2 / scale0[0], y0 * 2 / scale0[1], x1 * 2 / scale1[0], y1 * 2 / scale1[1]
        
        # 存储线段信息
        lines.append(((int(x0), int(y0)), (int(x1) + W1, int(y1)), color))
        
        # 存储圆圈信息
        circles.append(((int(x0), int(y0)), color))
        circles.append(((int(x1) + W1, int(y1)), color))

    # 先绘制所有线段
    for start, end, color in lines:
        cv2.line(combined_image, start, end, color, 1, lineType=cv2.LINE_AA)

    # 再绘制所有圆圈
    for center, color in circles:
        cv2.circle(combined_image, center, 2, color, -1, lineType=cv2.LINE_AA)

    # 保存图像到文件
    output_path = 'sample_match.png'
    cv2.imwrite(output_path, combined_image)

    # 使用 matplotlib 显示拼接后的图像
    plt.imshow(cv2.cvtColor(combined_image, cv2.COLOR_BGR2RGB))
    plt.axis('off')  # 关闭坐标轴
    plt.show()    

In [ ]:
t_errs = []
R_errs = []
epi_errs = []
fine_coord_0s = []
fine_coord_1s = []
coarse_coord_1s = []
target_fine_coord_0s = []
target_fine_coord_1s = []
target_coarse_coord_1s = []

for data in datas:
    data = recursive_to(data, device)
    with torch.no_grad():
        maff(data)
        torch.cuda.empty_cache()
    coarse_coord_0 = data["coarse_coord_0"]
    coarse_coord_1 = data["coarse_coord_1"]
    fine_coord_0 = data["fine_coord_0"].clone().detach()
    fine_coord_1 = data["fine_coord_1"].clone().detach()
    compute_supervision_coarse(data, coarse_scale=config.MODEL.COARSE_SCALE)
    W = data["image0"].shape[-1] / config.MODEL.COARSE_SCALE
    scale = (
        1
        if config["MODEL"]["PIXEL_SHUFFLE_REFINEMENT"]
        else config["MODEL"]["FINE_SCALE"]
    )
    scale0 = data["scale0"] if "scale0" in data else 1
    scale1 = data["scale1"] if "scale1" in data else 1
    # coarse_coord_0 = torch.tensor([[i%W + 0.5,i//W + 0.5] for i in data["spv_i_ids"]], device=device)
    # coarse_coord_1 = torch.tensor([[i%W + 0.5,i//W + 0.5] for i in data["spv_j_ids"]], device=device)
    # coarse_coord_0 = torch.tensor([[i%W,i//W] for i in data["spv_i_ids"]], device=device)
    # coarse_coord_1 = torch.tensor([[i%W,i//W] for i in data["spv_j_ids"]], device=device)
    target_coarse_coord_0 = torch.tensor([[i%W,i//W] for i in data["i_idx_c"]], device=device)
    target_coarse_coord_1 = torch.tensor([[i%W,i//W] for i in data["j_idx_c"]], device=device)
    
    # data.update({
    #     "b_idx_c": data["spv_b_ids"],
    #     "i_idx_c": data["spv_i_ids"],
    #     "j_idx_c": data["spv_j_ids"],
    # })
    
    compute_supervision_fine(data, config)
    
    # fine_coord_0 = coarse_coord_0 * config.MODEL.COARSE_SCALE * scale0
    target_fine_coord_0 = target_coarse_coord_0 * config.MODEL.COARSE_SCALE * scale0
    # 添加随机扰动
    random_noise = torch.randn_like(data["expec_f_gt"], device=device) * 0.  # 0.1 是扰动的强度，可以根据需要调整
    fixed_noise = torch.tensor([[0.0, 0.0]], device=device).repeat(data["expec_f_gt"].shape[0],1)
    radius = (
        config["MODEL"]["COARSE_SCALE"] // 2
        if config["MODEL"]["PIXEL_SHUFFLE_REFINEMENT"]
        else int(config["MODEL"]["COARSE_SCALE"] // config["MODEL"]["FINE_SCALE"]) // 2
    )
    mask = torch.linalg.norm(data["expec_f_gt"], ord=float("inf"), dim=1) < 1.0
    data["expec_f_gt"][~mask] = 0
    # fine_coord_1 = coarse_coord_1 * config.MODEL.COARSE_SCALE * scale1 + (data["expec_f_gt"] + random_noise + fixed_noise) * scale1 * scale * radius
    target_fine_coord_1 = target_coarse_coord_1 * config.MODEL.COARSE_SCALE * scale1 + (data["expec_f_gt"] + random_noise + fixed_noise) * scale1 * scale * radius
    
    fine_coord_0s.append(fine_coord_0[mask])
    fine_coord_1s.append(fine_coord_1[mask])
    target_fine_coord_0s.append(target_fine_coord_0[mask])
    target_fine_coord_1s.append(target_fine_coord_1[mask])
    coarse_coord_1s.append(coarse_coord_1[mask])
    target_coarse_coord_1s.append(target_coarse_coord_1[mask])
    # fine_coord_1 = coarse_coord_1 * config.MODEL.COARSE_SCALE * scale1
    # data.update({"coarse_coord_0": coarse_coord_0, "coarse_coord_1": coarse_coord_1, "fine_coord_0": fine_coord_0, "fine_coord_1": fine_coord_1})

    # # 从 data 中获取图像并转换为 numpy 数组
    # image0 = data["image0"].detach().cpu().numpy()[0, 0] * 255
    # W1 = data["image0"].shape[-1] 
    # image1 = data["image1"].detach().cpu().numpy()[0, 0] * 255

    # # 确保图像数据类型为 uint8
    # image0 = image0.astype(np.uint8)
    # image1 = image1.astype(np.uint8)

    # # 将灰度图像转换为 BGR 图像
    # image0 = cv2.cvtColor(image0, cv2.COLOR_GRAY2BGR)
    # image1 = cv2.cvtColor(image1, cv2.COLOR_GRAY2BGR)

    # # 将图像插值至两倍大小
    # image0 = cv2.resize(image0, (image0.shape[1] * 2, image0.shape[0] * 2), interpolation=cv2.INTER_LINEAR)
    # image1 = cv2.resize(image1, (image1.shape[1] * 2, image1.shape[0] * 2), interpolation=cv2.INTER_LINEAR)
    # W1 *= 2

    # # 将两张图像水平拼接在一起
    # combined_image = cv2.hconcat([image0, image1])

    # # 预定义一组常规的颜色
    # colors = [
    #     (255, 0, 0),    # 红色
    #     (0, 255, 0),    # 绿色
    #     (0, 0, 255),    # 蓝色
    #     (255, 255, 0),  # 黄色
    #     (255, 0, 255),  # 洋红
    #     (0, 255, 255),  # 青色
    #     (128, 0, 0),    # 深红
    #     (0, 128, 0),    # 深绿
    #     (0, 0, 128),    # 深蓝
    #     (128, 128, 0),  # 橄榄绿
    #     (128, 0, 128),  # 紫色
    #     (0, 128, 128),  # 青绿色
    #     (192, 192, 192),# 银色
    #     (128, 128, 128),# 灰色
    #     (0, 0, 0),      # 黑色
    #     (255, 165, 0),  # 橙色
    #     (255, 20, 147), # 深粉色
    #     (75, 0, 130),   # 靛蓝色
    #     (240, 230, 140),# 卡其色
    #     (173, 216, 230) # 淡蓝色
    # ]

    # # 在图像上绘制连线和圆圈
    # lines = []
    # circles = []

    # for idx, ((x0, y0), (x1, y1)) in enumerate(zip(fine_coord_0, fine_coord_1)):
    #     # 从预定义的颜色中随机选择一个颜色
    #     color = (np.random.randint(0, 256), np.random.randint(0, 256), np.random.randint(0, 256))
        
    #     scale0 = data["scale0"][0] if "scale0" in data else 1
    #     scale1 = data["scale1"][0] if "scale1" in data else 1
        
    #     # 调整坐标至两倍大小
    #     x0, y0, x1, y1 = x0 * 2 / scale0[0], y0 * 2 / scale0[1], x1 * 2 / scale1[0], y1 * 2 / scale1[1]
        
    #     # 存储线段信息
    #     lines.append(((int(x0), int(y0)), (int(x1) + W1, int(y1)), color))
        
    #     # 存储圆圈信息
    #     circles.append(((int(x0), int(y0)), color))
    #     circles.append(((int(x1) + W1, int(y1)), color))

    # # 先绘制所有线段
    # for start, end, color in lines:
    #     cv2.line(combined_image, start, end, color, 1, lineType=cv2.LINE_AA)

    # # 再绘制所有圆圈
    # for center, color in circles:
    #     cv2.circle(combined_image, center, 2, color, -1, lineType=cv2.LINE_AA)

    # # 保存图像到文件
    # output_path = 'sample_match.png'
    # cv2.imwrite(output_path, combined_image)

    # # 使用 matplotlib 显示拼接后的图像
    # plt.imshow(cv2.cvtColor(combined_image, cv2.COLOR_BGR2RGB))
    # plt.axis('off')  # 关闭坐标轴
    # plt.show()    
    
    b_idx_c = torch.zeros(len(fine_coord_0), dtype=torch.int32)
    data.update({"b_idx_c": b_idx_c})
    compute_symmetrical_epipolar_errors(data)
    compute_pose_errors(data, config)
    # print(f"t_errs: {data['t_errs']}\nR_errs: {data['R_errs']}")
    t_errs.append(data['t_errs'])
    R_errs.append(data['R_errs'])
    epi_errs.append(data['epi_errs'])
    torch.cuda.empty_cache()

# Print results
t_errs = torch.tensor(t_errs).squeeze()
R_errs = torch.tensor(R_errs).squeeze()
aucs = [5, 10, 20]
thres = 1e-4
R_auc = compute_auc(R_errs, aucs)
t_auc = compute_auc(t_errs, aucs)
combined_errs = torch.max(R_errs, t_errs)
combined_auc = compute_auc(combined_errs, aucs)
print("\nR AUC:")
for k, v in R_auc.items():
    print(f"{k}: {v:.4f}")
print("\nt AUC")
for k, v in t_auc.items():
    print(f"{k}: {v:.4f}")
print("\nAUC:")
for k, v in combined_auc.items():
    print(f"{k}: {v:.4f}")


correct = 0
total = 0
for i in range(len(epi_errs)):
    correct += (epi_errs[i] < thres).sum()
    total += len(epi_errs[i])
print(f"prec@{thres: e}: \t{((correct / total).item()) * 100 : .4f}")

In [ ]:
fine_coord_0s = torch.concat(fine_coord_0s, dim=0)
fine_coord_1s = torch.concat(fine_coord_1s, dim=0)
coarse_coord_1s = torch.concat(coarse_coord_1s, dim=0)
target_fine_coord_0s = torch.concat(target_fine_coord_0s, dim=0)
target_fine_coord_1s = torch.concat(target_fine_coord_1s, dim=0)
target_coarse_coord_1s = torch.concat(target_coarse_coord_1s, dim=0)

In [ ]:
diff = (fine_coord_1s - coarse_coord_1s)
diff_x = diff[:,0] ** 2
diff_y = diff[:,1] ** 2
print(f"diff_x: {diff_x.mean()}\ndiff_y: {diff_y.mean()}")

In [ ]:
diff = (fine_coord_1s - target_fine_coord_1s)
diff_x = diff[:,0] ** 2
diff_y = diff[:,1] ** 2
print(f"diff_x: {diff_x.mean()}\ndiff_y: {diff_y.mean()}")

In [ ]:
data["expec_f"][0]

In [ ]:
with torch.no_grad():
    compute_supervision_coarse(data, coarse_scale=config.MODEL.COARSE_SCALE)
    maff(data)
    # compute epi_errs for each match
    compute_symmetrical_epipolar_errors(data)
    # compute R_errs, t_errs, pose_errs for each pair
    compute_pose_errors(data, config)
    rel_pair_names = list(zip(*data["pair_names"]))
    bs = data["image0"].size(0)
    metrics = {
        # to filter duplicate pairs caused by DistributedSampler
        "identifiers": ["#".join(rel_pair_names[b]) for b in range(bs)],
        "epi_errs": [
            data["epi_errs"][data["b_idx_c"] == b].cpu().numpy() for b in range(bs)
        ],
        "R_errs": data["R_errs"],
        "t_errs": data["t_errs"],
        "inliers": data["inliers"],
    }
    val_metrics_4tb = aggregate_metrics(metrics, 1e-4)

In [ ]:
print(f"t_errs: {data['t_errs']}\nR_errs: {data['R_errs']}\nInliers: {data['inliers']}")

In [ ]:
coarse_coord_0 = data["coarse_coord_0"]
coarse_coord_1 = data["coarse_coord_1"]
fine_coord_0 = data["fine_coord_0"]
fine_coord_1 = data["fine_coord_1"]

In [ ]:
# 从 data 中获取图像并转换为 numpy 数组
image0 = data["image0"].detach().cpu().numpy()[0, 0] * 255
W1 = data["hw0_i"][1]
image1 = data["image1"].detach().cpu().numpy()[0, 0] * 255

# 确保图像数据类型为 uint8
image0 = image0.astype(np.uint8)
image1 = image1.astype(np.uint8)

# 将灰度图像转换为 BGR 图像
image0 = cv2.cvtColor(image0, cv2.COLOR_GRAY2BGR)
image1 = cv2.cvtColor(image1, cv2.COLOR_GRAY2BGR)

# 将图像插值至两倍大小
image0 = cv2.resize(image0, (image0.shape[1] * 2, image0.shape[0] * 2), interpolation=cv2.INTER_LINEAR)
image1 = cv2.resize(image1, (image1.shape[1] * 2, image1.shape[0] * 2), interpolation=cv2.INTER_LINEAR)
W1 *= 2

# 将两张图像水平拼接在一起
combined_image = cv2.hconcat([image0, image1])

# 预定义一组常规的颜色
colors = [
    (255, 0, 0),    # 红色
    (0, 255, 0),    # 绿色
    (0, 0, 255),    # 蓝色
    (255, 255, 0),  # 黄色
    (255, 0, 255),  # 洋红
    (0, 255, 255),  # 青色
    (128, 0, 0),    # 深红
    (0, 128, 0),    # 深绿
    (0, 0, 128),    # 深蓝
    (128, 128, 0),  # 橄榄绿
    (128, 0, 128),  # 紫色
    (0, 128, 128),  # 青绿色
    (192, 192, 192),# 银色
    (128, 128, 128),# 灰色
    (0, 0, 0),      # 黑色
    (255, 165, 0),  # 橙色
    (255, 20, 147), # 深粉色
    (75, 0, 130),   # 靛蓝色
    (240, 230, 140),# 卡其色
    (173, 216, 230) # 淡蓝色
]

# 在图像上绘制连线和圆圈
lines = []
circles = []

for idx, ((x0, y0), (x1, y1)) in enumerate(zip(fine_coord_0, fine_coord_1)):
    if data["inliers"][0][idx] == 0:
        continue
    # 从预定义的颜色中随机选择一个颜色
    color = (np.random.randint(0, 256), np.random.randint(0, 256), np.random.randint(0, 256))
    
    scale0 = data["scale0"][0] if "scale0" in data else 1
    scale1 = data["scale1"][0] if "scale1" in data else 1
    
    # 调整坐标至两倍大小
    x0, y0, x1, y1 = x0 * 2 / scale0[0], y0 * 2 / scale0[1], x1 * 2 / scale1[0], y1 * 2 / scale1[1]
    
    # 存储线段信息
    lines.append(((int(x0), int(y0)), (int(x1) + W1, int(y1)), color))
    
    # 存储圆圈信息
    circles.append(((int(x0), int(y0)), color))
    circles.append(((int(x1) + W1, int(y1)), color))

# 先绘制所有线段
for start, end, color in lines:
    cv2.line(combined_image, start, end, color, 1, lineType=cv2.LINE_AA)

# 再绘制所有圆圈
for center, color in circles:
    cv2.circle(combined_image, center, 2, color, -1, lineType=cv2.LINE_AA)

# 保存图像到文件
output_path = 'sample_match.png'
cv2.imwrite(output_path, combined_image)

# 使用 matplotlib 显示拼接后的图像
plt.imshow(cv2.cvtColor(combined_image, cv2.COLOR_BGR2RGB))
plt.axis('off')  # 关闭坐标轴
plt.show()

In [ ]:
from utils.misc import setup_gpus
import torch
from maff.backbone import build_backbone
from default_config import get_cfg_defaults

config = get_cfg_defaults()
config.MODEL.BACKBONE.BACKBONE_TYPE = "VMamba_T_FPN"
n_gpu_available = setup_gpus("7,")
device = torch.device("cuda:0")

model = build_backbone(config["MODEL"]["BACKBONE"]).to(device)

In [ ]:
x = torch.randn(1, 1, 640, 640).to(device)
features = model(x)

In [ ]:
[print(f.shape) for f in features]

In [ ]:
def _extract_feat_window(
    _feat: torch.Tensor,
    b_idx: torch.Tensor,
    coord: torch.Tensor,
    window_size: int,
):
    # Calculate row and column indices
    row_indices = coord[:, 1].long()
    col_indices = coord[:, 0].long()

    # Create grid indices
    row_offsets = torch.arange(window_size, device=_feat.device, dtype=torch.long)
    col_offsets = torch.arange(window_size, device=_feat.device, dtype=torch.long)

    row_indices = row_indices.unsqueeze(1) + row_offsets.unsqueeze(0)
    col_indices = col_indices.unsqueeze(1) + col_offsets.unsqueeze(0)

    # Extract windows using advanced indexing
    windows = _feat[
        b_idx[:, None, None], :, row_indices[:, :, None], col_indices[:, None, :]
    ]

    # Rearrange to [B, C, H, W]
    windows = rearrange(windows, "b h w c -> b c h w")

    return windows

In [ ]:
import torch.nn as nn

class DualMultiScaleSinePositionalEncoding(nn.Module):
    @torch.no_grad
    def __init__(
        self,
        d_model: int,
        max_hw: int,
        scales: Sequence[float],
        dtype: torch.dtype = torch.float32,
    ):
        """
        Default Constructor for dual input multi-scale sinusoidal positional encoding, alike 3D PE, make sure features in all scales are transform into same channel number before input.
        Args:
            d_model (int): Typically the largest dimensions of features in all scale, e.g. for ResNet18: 512
            max_hw (int): the largest length of both side of image input. MAX(maximum_height, maximum_width)
            scales (Sequence[int]): Scales of features, e.g. for ResNet18: [0.5, 0.25, 0.125, 0.0625]
            dtype (torch.dtype): Data type, must be in float type
        """
        super(DualMultiScaleSinePositionalEncoding, self).__init__()

        self.d_model = d_model
        self.max_hw = max_hw
        self.scales = scales
        self.dtype = dtype
        self.num_scales = len(scales)

        # build multi-scale position embedding map
        self.all_scales_pos_emb = []  # scale, C, H, W

        pos_x = torch.arange(self.max_hw, dtype=self.dtype)
        pos_y = torch.arange(self.max_hw, dtype=self.dtype)
        pos_z = torch.arange(2 * self.num_scales, dtype=self.dtype)

        d_pos_embeddings = int(np.ceil(self.d_model / 3))
        inv_freq = 1.0 / (
            10000.0
            ** (
                torch.arange(0, d_pos_embeddings, 2, dtype=self.dtype)
                / d_pos_embeddings
            )
        )

        hw_pos_emb = torch.zeros(
            size=(self.max_hw, self.max_hw, 6 * inv_freq.shape[0]), dtype=self.dtype
        )  # H, W, C

        for input in range(2):
            for i in range(self.num_scales):
                temp_x = torch.einsum("i,j->ij", pos_x, inv_freq)
                temp_y = torch.einsum("i,j->ij", pos_y, inv_freq)
                temp_z = pos_z[i + input * self.num_scales] * inv_freq

                sin_x = torch.sin(temp_x).unsqueeze(0)
                sin_y = torch.sin(temp_y).unsqueeze(1)
                sin_z = torch.sin(temp_z).unsqueeze(0).unsqueeze(0)
                cos_x = torch.cos(temp_x).unsqueeze(0)
                cos_y = torch.cos(temp_y).unsqueeze(1)
                cos_z = torch.cos(temp_z).unsqueeze(0).unsqueeze(0)

                hw_pos_emb[:, :, 0::6] = sin_x
                hw_pos_emb[:, :, 1::6] = cos_x
                hw_pos_emb[:, :, 2::6] = sin_y
                hw_pos_emb[:, :, 3::6] = cos_y
                hw_pos_emb[:, :, 4::6] = sin_z
                hw_pos_emb[:, :, 5::6] = cos_z

                # Crop the exceeding channels
                hw_pos_emb_cropped = hw_pos_emb[:, :, 0 : self.d_model]

                # H, W, C -> C, H, W
                hw_pos_emb_cropped = hw_pos_emb_cropped.swapaxes(0, -1).swapaxes(-1, -2)

                # Interpolate into each scales
                hw_pos_emb_rescaled = F.interpolate(
                    hw_pos_emb_cropped.unsqueeze(0),
                    scale_factor=(scales[i], scales[i]),
                ).squeeze(0)

                self.register_buffer(
                    f"all_scales_pos_emb_{i + input * self.num_scales}",
                    hw_pos_emb_rescaled,
                    persistent=False,
                )

    def forward(self, x1: Sequence[torch.Tensor], x2: Sequence[torch.Tensor]):
        """
        Add PE into multi_scale_input. Make sure the scaling and number of channels is same as when you construct this PE
        Args:
            x1 (Sequence[torch.Tensor]): Multi-scale input 1: Scales x tensor[B x C x H x W], make sure features in all scales are transform into same channel number before input.
            x2 (Sequence[torch.Tensor]): Multi-scale input 2: Scales x tensor[B x C x H x W], make sure features in all scales are transform into same channel number before input.
        """
        for s, single_scale in enumerate(x1):
            batch_size = single_scale.shape[0]
            for b in range(batch_size):
                single_scale[b] = single_scale[b] + getattr(
                    self, f"all_scales_pos_emb_{s}"
                )

        for s, single_scale in enumerate(x2):
            batch_size = single_scale.shape[0]
            for b in range(batch_size):
                single_scale[b] = single_scale[b] + getattr(
                    self, f"all_scales_pos_emb_{s + self.num_scales}"
                )
        return x1, x2

    def get_position_encodings(self):
        """
        返回所有尺度的位置编码
        """
        encodings = []
        for i in range(2 * self.num_scales):
            encodings.append(getattr(self, f"all_scales_pos_emb_{i}"))
        return encodings

In [ ]:
from utils.augment import build_augmentor
from datasets.utils.dataset import imread_gray
import cv2

augmentor = build_augmentor("maff")
original_image = imread_gray("sample_match.png")
original_image = cv2.cvtColor(original_image, cv2.COLOR_GRAY2RGB)
augmented_image = imread_gray("sample_match.png", augmentor)
augmented_image = cv2.cvtColor(augmented_image, cv2.COLOR_GRAY2RGB)

plt.imshow(original_image)
plt.axis('off')  # 关闭坐标轴
plt.show()

In [ ]:
plt.imshow(augmented_image)
plt.axis('off')  # 关闭坐标轴
plt.show()